# Stage 10C — GSE270828 low-memory pseudobulk (NO personal Google Drive)

**Purpose:** Get original integer RNA counts for GSE270828 by reading a public processed `.h5ad` on disk in small chunks. **No `drive.mount`, sign-in to Google Drive, Seurat, or complete 2.31-GB RDS deserialization.**

This notebook attempts to find *one* GSE270828 `.h5ad` in PRISM's public file listing. A public host may be Google Drive, but **no access to your personal Google Drive is requested**. If listing is blocked, the output ZIP includes a precise reason and you can specify a public single-file URL or local `/content/*.h5ad` path in the configuration below.

**Safety:** Analyze only experimentally observed integer counts. Stop if selected expression is not raw integer count data. Output remains a pseudobulk descriptive count matrix: **not differential expression, not causal validation**. Replicate labels require independent-design verification.

In [ ]:
%pip -q install "gdown>=6.1,<7" "h5py>=3.9" "scipy>=1.10" "pandas>=2.0" "numpy>=1.24"

In [ ]:
# Configuration — this does not connect your personal Google Drive.
# Leave defaults to try the public PRISM file listing for GSE270828.
PUBLIC_PRISM_FOLDER_ID = "1Y0Z19JhiTmTch65kvBNNMdVtosH6QHfi"
DIRECT_PUBLIC_FILE_ID = ""     # Optional: the ID of a PUBLIC, single .h5ad file
DIRECT_PUBLIC_URL = ""         # Optional: direct, PUBLIC https URL to one .h5ad file
LOCAL_H5AD_PATH = ""           # Optional: e.g. /content/GSE270828_merged.h5ad
CHUNK_CELLS = 256              # 128 if low RAM; 512 if high RAM

from pathlib import Path
ROOT = Path("/content/HAR_Stage10C")
OUT = ROOT / "results"
ROOT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)

In [ ]:
# Self-contained analysis script — no extra file needs to be uploaded.
from pathlib import Path
core_code = '"""GSE270828 h5ad disk-backed pseudobulk aggregator (no Seurat, no Drive mount).\n\nInput: publicly available, downloaded PRISM GSE270828 .h5ad. Only raw, integer\nRNA counts are accepted. H5AD X/layers/raw read directly through h5py.\nThis script does NOT perform differential expression or causal inference.\n"""\nfrom __future__ import annotations\nimport csv, gzip, json, os, sys, traceback\nfrom collections import Counter\nfrom pathlib import Path\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\n\n\ndef decode_string_array(data):\n    return np.asarray([v.decode(\'utf-8\',errors=\'replace\') if isinstance(v,bytes) else str(v) for v in data], dtype=str)\n\n\ndef read_obs_column(g, name):\n    if name not in g:\n        raise KeyError(f"Missing required cell metadata field: {name}")\n    obj = g[name]\n    if isinstance(obj, h5py.Group) and \'codes\' in obj and \'categories\' in obj:\n        codes = obj[\'codes\'][:]\n        cats = decode_string_array(obj[\'categories\'][:])\n        return np.asarray([cats[int(c)] if 0 <= int(c) < len(cats) else \'\' for c in codes])\n    if isinstance(obj,h5py.Dataset):\n        return decode_string_array(obj[:])\n    raise ValueError(f\'Unsupported h5ad obs field: {name}\')\n\n\ndef read_genes(f, group_key, n_genes):\n    vg = f[\'raw/var\'] if group_key.startswith(\'raw/\') and \'raw/var\' in f else f[\'var\']\n    idxname = vg.attrs.get(\'_index\',\'_index\')\n    if isinstance(idxname,bytes): idxname=idxname.decode()\n    if idxname not in vg: idxname=\'_index\'\n    if idxname not in vg:\n        raise ValueError(f\'Gene IDs not found under {vg.name}\')\n    genes = decode_string_array(vg[idxname][:])\n    if len(genes) != n_genes: raise ValueError(\'Gene IDs / matrix dimensions mismatch\')\n    if len(set(genes)) != len(genes):\n        raise ValueError(\'Duplicated gene identifiers; manual resolution required before aggregation\')\n    return genes\n\n\ndef matrix_shape(x):\n    if isinstance(x,h5py.Dataset): return tuple(x.shape)\n    shape = x.attrs.get(\'shape\')\n    if shape is None and \'shape\' in x: shape=x[\'shape\'][:]\n    if shape is None: raise ValueError(f\'Missing sparse shape at {x.name}\')\n    return tuple(int(y) for y in shape)\n\n\ndef read_rows(x,start,end):\n    if isinstance(x,h5py.Dataset):\n        return np.asarray(x[start:end,:])\n    enc=x.attrs.get(\'encoding-type\',\'\')\n    if isinstance(enc,bytes): enc=enc.decode()\n    if enc != \'csr_matrix\':\n        raise ValueError(f\'{x.name}: {enc!r} encoding is not row-streamable with this code. Need CSR h5ad.\')\n    p=x[\'indptr\'][start:end+1]\n    off=int(p[0]); last=int(p[-1])\n    values=x[\'data\'][off:last]\n    cols=x[\'indices\'][off:last]\n    return sparse.csr_matrix((values,cols,p-off),shape=(end-start,matrix_shape(x)[1]))\n\n\ndef read_gene_columns_csc(x,start,end):\n    if isinstance(x,h5py.Dataset):return np.asarray(x[:,start:end])\n    enc=x.attrs.get(\'encoding-type\',\'\')\n    if isinstance(enc,bytes):enc=enc.decode()\n    if enc!=\'csc_matrix\':raise ValueError(\'CSC reader requires csc_matrix\')\n    p=x[\'indptr\'][start:end+1]\n    offset=int(p[0]);last=int(p[-1])\n    return sparse.csc_matrix((x[\'data\'][offset:last],x[\'indices\'][offset:last],p-offset),shape=(matrix_shape(x)[0],end-start))\n\n\ndef is_valid_count_block(mat):\n    arr = mat.data if sparse.issparse(mat) else np.asarray(mat)\n    return bool(np.isfinite(arr).all() and (arr>=0).all() and (np.abs(arr-np.rint(arr))<1e-6).all())\n\n\ndef choose_count_matrix(f,n_cells):\n    candidates=[\'layers/counts\',\'layers/raw_counts\',\'raw/X\',\'X\']\n    report=[]\n    for key in candidates:\n        if key not in f: continue\n        x=f[key]\n        try:\n            sh=matrix_shape(x)\n            if len(sh)!=2 or sh[0]!=n_cells: raise ValueError(f\'shape {sh} vs {n_cells} cells\')\n            if isinstance(x,h5py.Group):\n                enc=x.attrs.get(\'encoding-type\',\'\')\n                if isinstance(enc,bytes): enc=enc.decode()\n                if enc not in (\'csr_matrix\',\'csc_matrix\'): raise ValueError(f\'only CSR/CSC supported, found {enc}\')\n            # Sample multiple locations; every value is validated during aggregation.\n            ok=True\n            if isinstance(x,h5py.Group) and enc==\'csc_matrix\':\n                for st in sorted(set([0,max(0,sh[1]//3),max(0,2*sh[1]//3),max(0,sh[1]-16)])):\n                    if not is_valid_count_block(read_gene_columns_csc(x,st,min(st+16,sh[1]))):ok=False;break\n            else:\n                for st in sorted(set([0,max(0,n_cells//3),max(0,2*n_cells//3),max(0,n_cells-16)])):\n                    if not is_valid_count_block(read_rows(x,st,min(st+16,n_cells))):ok=False;break\n            report.append({\'key\':key,\'shape\':list(sh),\'looks_like_raw_counts\':ok})\n            if ok:return key,x,report\n        except Exception as exc:\n            report.append({\'key\':key,\'error\':str(exc)})\n    raise ValueError(\'No verified raw-count matrix found. Candidate inspection: \'+repr(report))\n\n\ndef aggregate_h5ad(h5ad_path,output_dir,chunk_size=256):\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    result={\'status\':\'started\',\'source\':str(h5ad_path),\'computes_p_values\':False,\'biological_replication_assumed\':False}\n    with h5py.File(h5ad_path,\'r\') as f:\n        if \'obs\' not in f:raise ValueError(\'Not a valid AnnData file: obs missing\')\n        obs=f[\'obs\']\n        rep=read_obs_column(obs,\'rep\') if \'rep\' in obs else read_obs_column(obs,\'replicate\')\n        har=read_obs_column(obs,\'har\')\n        numguide=read_obs_column(obs,\'num_guide\')\n        guides=read_obs_column(obs,\'guide\') if \'guide\' in obs else np.array([\'\']*len(har))\n        n_cells=len(har)\n        if n_cells<1000:raise ValueError(\'Expected full GSE270828 object; too few cells\')\n        key,x,source_report=choose_count_matrix(f,n_cells)\n        n_rows,n_genes=matrix_shape(x)\n        genes=read_genes(f,key,n_genes)\n        keep=(numguide==\'single_true\') & (rep!=\'\') & (har!=\'\')\n        if keep.sum()<1000:raise ValueError(\'Not enough single_true cells\')\n        label_all=np.char.add(np.char.add(rep.astype(str),\'__\'),har.astype(str))\n        groups=sorted(set(label_all[keep].tolist()))\n        gidx={k:i for i,k in enumerate(groups)}\n        membership=np.full(n_cells,-1,dtype=np.int32)\n        membership[keep]=np.array([gidx[x] for x in label_all[keep]],dtype=np.int32)\n        agg=np.zeros((len(groups),n_genes),dtype=np.int64)\n        count_by_group=np.zeros(len(groups),dtype=np.int64)\n        enc = x.attrs.get(\'encoding-type\',\'\') if isinstance(x,h5py.Group) else \'dense\'\n        if isinstance(enc,bytes):enc=enc.decode()\n        if enc==\'csc_matrix\':\n            # Column-streaming for CSC: no full matrix or densification of cells x genes.\n            selected_indices=np.where(keep)[0]\n            selected_groups=membership[selected_indices]\n            M_all=sparse.csr_matrix((np.ones(len(selected_groups),dtype=np.int64),(selected_groups,selected_indices)),shape=(len(groups),n_cells))\n            count_by_group=np.bincount(selected_groups,minlength=len(groups)).astype(np.int64)\n            gene_chunk=min(max(16,chunk_size//2),128)\n            for first in range(0,n_genes,gene_chunk):\n                last=min(first+gene_chunk,n_genes)\n                chunk=read_gene_columns_csc(x,first,last)\n                if not is_valid_count_block(chunk):\n                    raise ValueError(f\'Noninteger or invalid expression: {key}, gene columns {first}:{last}. Pseudobulk REFUSED.\')\n                block=(M_all @ chunk).toarray()\n                if not is_valid_count_block(block):raise ValueError(\'Invalid CSC aggregated counts\')\n                agg[:,first:last]=np.rint(block).astype(np.int64)\n                if first%(gene_chunk*50)==0:print(f\'Processed {last:,}/{n_genes:,} genes (CSC)\',flush=True)\n        else:\n            for start in range(0,n_cells,chunk_size):\n                stop=min(start+chunk_size,n_cells)\n                local=membership[start:stop]\n                mat=read_rows(x,start,stop)\n                if not is_valid_count_block(mat):\n                    raise ValueError(f\'Noninteger, nonfinite, or negative expression: {key}, rows {start}:{stop}. Pseudobulk REFUSED.\')\n                valid=local>=0\n                if not valid.any():continue\n                ids=local[valid]\n                count_by_group += np.bincount(ids,minlength=len(groups))\n                selected=mat[valid,:]\n                if not sparse.issparse(selected):selected=sparse.csr_matrix(selected)\n                M=sparse.csr_matrix((np.ones(len(ids),dtype=np.int64),(ids,np.arange(len(ids)))),shape=(len(groups),len(ids)))\n                block=(M @ selected).toarray()\n                if not is_valid_count_block(block):raise ValueError(\'Aggregate count noninteger/invalid\')\n                agg += np.rint(block).astype(np.int64)\n                if start%(chunk_size*50)==0:\n                    print(f\'Processed {stop:,}/{n_cells:,} cells; selected={count_by_group.sum():,}\',flush=True)\n        if int(count_by_group.sum())!=int(keep.sum()):raise AssertionError(\'Selected cell count mismatch\')\n        # Verify metadata nCount_RNA if present and represents raw RNA assay.\n        umi_check={}\n        if \'nCount_RNA\' in obs and isinstance(obs[\'nCount_RNA\'],h5py.Dataset):\n            try:\n                original=np.asarray(obs[\'nCount_RNA\'][:],dtype=float)\n                agg_total=int(agg.sum())\n                obs_total=float(original[keep].sum())\n                umi_check={\'sum_pseudobulk_raw_counts\':agg_total,\'sum_metadata_nCount_RNA\':obs_total,\'relative_difference\':float(abs(agg_total-obs_total)/max(obs_total,1))}\n            except Exception as exc:umi_check={\'error\':str(exc)}\n        meta=[]\n        for k,g in enumerate(groups):\n            re,ha=g.split(\'__\',1)\n            meta.append({\'group\':g,\'rep\':re,\'har\':ha,\'n_cells\':int(count_by_group[k]),\'control\':ha.casefold().startswith(\'non-target\') or ha.upper().startswith(\'NTC\')})\n        pd.DataFrame(meta).to_csv(output_dir/\'pseudobulk_group_metadata.csv\',index=False)\n        pd.DataFrame({\'gene_id\':genes}).to_csv(output_dir/\'gene_ids.csv\',index=False)\n        # Write the entire original-count aggregate; no top-gene truncation.\n        with gzip.open(output_dir/\'pseudobulk_RAW_COUNTS_all_genes.tsv.gz\',\'wt\',newline=\'\') as z:\n            writer=csv.writer(z,delimiter=\'\\t\')\n            writer.writerow([\'gene_id\',*groups])\n            for i,gene in enumerate(genes):writer.writerow([gene,*agg[:,i].tolist()])\n        result.update({\'status\':\'success\',\'n_cells_original\':int(n_cells),\'n_cells_selected\':int(keep.sum()),\'n_genes\':int(n_genes),\'n_groups\':int(len(groups)),\'n_har_labels\':len(set(har[keep].tolist())),\'n_replicate_labels\':len(set(rep[keep].tolist())),\'matrix_selected\':key,\'matrix_candidate_qc\':source_report,\'count_metadata_comparison\':umi_check,\'n_control_groups\':int(sum(row[\'control\'] for row in meta)),\'n_non_targeting_control_cells_selected\':int(sum(row[\'n_cells\'] for row in meta if row[\'control\']))})\n        return result\n\n\ndef synthetic_test(directory):\n    """Fixture-only validation. Never use synthetic fixture data as findings."""\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    src=directory/\'synthetic_fixture.h5ad\'\n    X=sparse.csr_matrix(np.asarray([[1,0,2,1],[4,0,1,0],[1,5,0,1],[0,2,0,0],[5,2,1,0],[2,3,4,0]]*200,dtype=np.int32))\n    reps=np.asarray([\'rep1\',\'rep1\',\'rep2\',\'rep2\',\'rep3\',\'rep3\']*200)\n    hars=np.asarray([\'HAR_A\',\'Non-Targeting\',\'HAR_A\',\'Non-Targeting\',\'HAR_A\',\'Non-Targeting\']*200)\n    true=np.asarray([\'single_true\']*len(hars))\n    def h5strings(g,key,values):g.create_dataset(key,data=np.asarray(values,dtype=h5py.string_dtype(\'utf-8\')))\n    with h5py.File(src,\'w\') as f:\n        obs=f.create_group(\'obs\');h5strings(obs,\'rep\',reps);h5strings(obs,\'har\',hars);h5strings(obs,\'num_guide\',true)\n        var=f.create_group(\'var\');h5strings(var,\'_index\',[\'g1\',\'g2\',\'g3\',\'g4\'])\n        x=f.create_group(\'X\');x.attrs[\'encoding-type\']=\'csr_matrix\';x.attrs[\'shape\']=(X.shape[0],X.shape[1])\n        x.create_dataset(\'data\',data=X.data);x.create_dataset(\'indices\',data=X.indices);x.create_dataset(\'indptr\',data=X.indptr)\n    r=aggregate_h5ad(src,directory/\'outputs\',chunk_size=17)\n    m=pd.read_csv(directory/\'outputs/pseudobulk_group_metadata.csv\')\n    assert r[\'n_cells_selected\']==1200 and r[\'n_groups\']==6 and int(m[\'n_cells\'].sum())==1200\n    with gzip.open(directory/\'outputs/pseudobulk_RAW_COUNTS_all_genes.tsv.gz\',\'rt\') as z:\n        rr=pd.read_csv(z,sep=\'\\t\')\n    assert len(rr)==4 and int(rr.drop(columns=[\'gene_id\']).to_numpy().sum())==int(X.sum())\n    return {\'synthetic_smoke_test\':\'PASS\',\'n_cells\':1200,\'n_genes\':4,\'n_groups\':6}\n\n\nif __name__==\'__main__\':\n    if len(sys.argv)>1 and sys.argv[1]==\'--smoke\':\n        print(json.dumps(synthetic_test(sys.argv[2] if len(sys.argv)>2 else \'/tmp/stage10c_test\'),indent=2))\n    else:\n        if len(sys.argv)<3:raise SystemExit(\'usage: python stream_h5ad_pseudobulk.py DATA.h5ad OUTDIR\')\n        try:\n            report=aggregate_h5ad(sys.argv[1],sys.argv[2]);print(json.dumps(report,indent=2))\n            Path(sys.argv[2],\'QC_and_STATUS.json\').write_text(json.dumps(report,indent=2))\n        except Exception as e:\n            out=Path(sys.argv[2]);out.mkdir(parents=True,exist_ok=True)\n            status={\'status\':\'failed\',\'error\':str(e),\'traceback\':traceback.format_exc(),\'fabricated_data\':False}\n            (out/\'QC_and_STATUS.json\').write_text(json.dumps(status,indent=2))\n            raise\n'
script_file = ROOT / "stream_h5ad_pseudobulk.py"
script_file.write_text(core_code, encoding="utf-8")
import subprocess,sys
result = subprocess.run([sys.executable,str(script_file),"--smoke",str(ROOT/"synthetic_test")], text=True, capture_output=True)
print(result.stdout[-1800:]); print(result.stderr[-1800:])
assert result.returncode == 0, "Synthetic test failed — refusing to run unverified analysis"
print("Synthetic test successful. Synthetic data will NOT appear in the scientific outputs.")

In [ ]:
# Discover public source. Download ONLY the matching GSE270828 file, never the entire PRISM folder.
import os, json, re, shutil, subprocess, traceback, urllib.request
import gdown
status={'stage':'10C','drive_mount_used':False,'differential_expression':False,'biological_donor_independence_verified':False}
manifest=[]
source_path=None
try:
    if LOCAL_H5AD_PATH.strip() and Path(LOCAL_H5AD_PATH).is_file():
        source_path=Path(LOCAL_H5AD_PATH)
        status['source_method']='local_h5ad'
    else:
        local_candidates = sorted(Path('/content').glob('*GSE270828*.h5ad'))
        if local_candidates:
            source_path=local_candidates[0]
            status['source_method']='existing_content_h5ad'
    if source_path is None and DIRECT_PUBLIC_FILE_ID.strip():
        p=ROOT/'GSE270828_merged.h5ad'
        ret=gdown.download(id=DIRECT_PUBLIC_FILE_ID.strip(),output=str(p),quiet=False,use_cookies=False)
        if ret and p.is_file():source_path=p;status['source_method']='public_single_file_id'
    if source_path is None and DIRECT_PUBLIC_URL.strip():
        from urllib.parse import urlparse
        url=DIRECT_PUBLIC_URL.strip()
        if not url.startswith('https://'):raise ValueError('Only HTTPS sources accepted')
        p=ROOT/'GSE270828_merged.h5ad'
        import requests
        with requests.get(url,stream=True,timeout=120) as r:
            r.raise_for_status()
            with open(p,'wb') as w:
                for b in r.iter_content(chunk_size=2*1024*1024):
                    if b:w.write(b)
        source_path=p;status['source_method']='public_https_url'
    if source_path is None:
        status['source_method']='public_prism_folder_discovery'
        print('Listing PUBLIC PRISM folder (no Google Drive account connection)...')
        listing=gdown.download_folder(id=PUBLIC_PRISM_FOLDER_ID,output=str(ROOT/'download'),quiet=True,use_cookies=False,remaining_ok=True,skip_download=True)
        for item in listing or []:
            manifest.append({'file_id':item.id,'path':item.path})
        import pandas as pd
        pd.DataFrame(manifest,columns=['file_id','path']).to_csv(OUT/'Public_PRISM_folder_inventory.csv',index=False)
        matches=[e for e in manifest if 'gse270828' in e['path'].lower() and e['path'].lower().endswith(('.h5ad','.h5ad.gz','.tar.gz'))]
        print(f'Listed {len(manifest)} public files, {len(matches)} GSE270828 candidate(s)')
        if matches:
            matches.sort(key=lambda x:(not x['path'].lower().endswith('.h5ad'),len(x['path'])))
            chosen=matches[0];status['selected_public_filename']=chosen['path']
            suffix='.h5ad' if chosen['path'].lower().endswith('.h5ad') else ('.h5ad.gz' if chosen['path'].lower().endswith('.h5ad.gz') else '.tar.gz')
            p=ROOT/('GSE270828_source'+suffix)
            print('Downloading only:',chosen['path'])
            got=gdown.download(id=chosen['file_id'],output=str(p),quiet=False,use_cookies=False)
            if got and p.exists():source_path=p
    if source_path is None:
        raise FileNotFoundError('No publicly accessible GSE270828 h5ad found. See Public_PRISM_folder_inventory.csv; provide DIRECT_PUBLIC_FILE_ID, DIRECT_PUBLIC_URL, or LOCAL_H5AD_PATH. Do not download full original RDS again in 12 GB RAM.')
    print('Downloaded/found source:',source_path,'bytes:',source_path.stat().st_size)
    # If the public download was a compressed h5ad, extract only the matching h5ad safely.
    import gzip,tarfile
    if str(source_path).endswith('.h5ad.gz'):
        dst=ROOT/'GSE270828_merged.h5ad'
        with gzip.open(source_path,'rb') as z,open(dst,'wb') as w:shutil.copyfileobj(z,w,length=4*1024*1024)
        source_path=dst
    elif str(source_path).endswith('.tar.gz'):
        dst=ROOT/'GSE270828_merged.h5ad'
        with tarfile.open(source_path,'r:gz') as t:
            hs=[m for m in t.getmembers() if m.isfile() and m.name.lower().endswith('.h5ad') and 'gse270828' in m.name.lower()]
            if len(hs)!=1:raise ValueError(f'Expected one matching h5ad inside archive, found {len(hs)}')
            with t.extractfile(hs[0]) as r,open(dst,'wb') as w:shutil.copyfileobj(r,w,length=4*1024*1024)
        source_path=dst
    import h5py
    if not h5py.is_hdf5(source_path):raise ValueError('Source is not a valid HDF5/h5ad file; check public source URL / download')
    status['h5ad_readable']=True
    status['file_bytes']=source_path.stat().st_size
    status['source_file']=str(source_path)
    print('HDF5 signature verified. Ready for streaming count aggregation.')
except Exception as exc:
    status['source_error']=str(exc)
    status['source_traceback']=traceback.format_exc()[-8000:]
    print('Source discovery limitation:',exc)
(OUT/'Stage10C_Source_Status.json').write_text(json.dumps(status,indent=2))

In [ ]:
# Streaming gene x replicate x HAR pseudobulk aggregation.
# No full expression matrix is loaded in memory, and no significance test is invented.
import subprocess,sys,json,traceback
if source_path is None:
    print("NOT RUN: Source h5ad unavailable. Source_Status.json describes next action.")
else:
    command=[sys.executable,str(script_file),str(source_path),str(OUT)]
    print('Starting 256-cell disk-backed aggregation (CHUNK_CELLS currently set to',CHUNK_CELLS,')')
    # Python wrapper calls function so CHUNK_CELLS can be set precisely.
    sys.path.insert(0,str(ROOT))
    from stream_h5ad_pseudobulk import aggregate_h5ad
    try:
        result=aggregate_h5ad(source_path,OUT,chunk_size=CHUNK_CELLS)
        print(json.dumps(result,indent=2))
        (OUT/'QC_and_STATUS.json').write_text(json.dumps(result,indent=2))
    except Exception as exc:
        status={'status':'failed_without_fake_results','error':str(exc),'traceback':traceback.format_exc(),'differential_expression':False}
        (OUT/'QC_and_STATUS.json').write_text(json.dumps(status,indent=2))
        print('Aggregation stopped:',str(exc))

In [ ]:
# Always export diagnostic/result ZIP to the user (even if public folder blocks discovery).
import zipfile,os,json
zip_file=Path('/content/HAR_Stage10C_Streaming_Results.zip')
with zipfile.ZipFile(zip_file,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,arcname=p.relative_to(OUT))
    z.write(script_file,arcname='stream_h5ad_pseudobulk.py')
print('Output:',zip_file, 'size MB:',round(zip_file.stat().st_size/1e6,2))
print('Output contents:')
with zipfile.ZipFile(zip_file) as z:print('\n'.join(z.namelist()[:25]))
from google.colab import files
files.download(str(zip_file))

## What to upload next
Upload `HAR_Stage10C_Streaming_Results.zip` to the same ChatGPT conversation. If it contains `pseudobulk_RAW_COUNTS_all_genes.tsv.gz`, the raw gene-count aggregation succeeded. If it contains only an inventory and status, the processed source was not publicly downloadable; the status explains the blocker.

**Do not interpret grouped cells, batches, or lanes as independent donors.** This step only aggregates counts; line/replicate-aware inference requires a separate statistical plan.